# ASN.1 grammar: parse and check RSA keys
Run `02_openssl_keys.ipynb` first – it creates the key files used here. Run cells from top to bottom. Modify only cells marked `TODO` and the grammar file; the check cells verify your work.

A DER file only contains anonymous INTEGERs. An ASN.1 grammar gives them names – and the names come with mathematical relations that we can check.

In [1]:
import os

for name in ["RSA_2048_priv.der", "RSA_2048_pub.der"]:
    assert os.path.exists(name), f"{name} is missing - run 02_openssl_keys.ipynb first"

### Task 1: Grammar and private key
- Look at the asn1tools example usage: https://asn1tools.readthedocs.io/en/latest/#example-usage
- The grammar of RSA keys is in [RFC 8017](https://www.rfc-editor.org/rfc/rfc8017), Appendix A.1.1 (`RSAPublicKey`) and A.1.2 (`RSAPrivateKey`) – the same definitions as in RFC 3447.
- **TODO:** complete the file `RSA_pub.asn` (open it from the Jupyter file browser), then run the cell. It compiles the grammar and decodes the RSA private key.

In [2]:
import asn1tools

rsa_spec = asn1tools.compile_files("RSA_pub.asn")
priv = rsa_spec.decode("RSAPrivateKey", open("RSA_2048_priv.der", "rb").read())
print(priv)
assert list(priv) == ["version", "modulus", "publicExponent", "privateExponent", "prime1", "prime2",
                      "exponent1", "exponent2", "coefficient"], "complete RSAPrivateKey in RSA_pub.asn"

{'version': 0, 'modulus': 26600849061124703356336481747273244556424897312346608964949093451607513266591471026106943762750980853446024141708766672109725982003015999854187390472741283703452960170203172706692751708465051997402750571528164091286451527513148700346006515152876039065521975773899338302200230413838645117361941719273038177957810503270043548045850626165166299090599599549006397916460291285774324441974192993549251294218087652252222592259968754112161696463425633580783656942059887637967565538581666544274937726011810584986873918823482565805634098210715585036832816990966938486266344067154502293851198053748263931313802845262670239577559, 'publicExponent': 3, 'privateExponent': 177338993740831355708909878315154963709499315415644059766327289677383421777276473507379625085006539022973494278058444480731506546686773332361249269818275224689686401134687818044618344723100346649351670476854427275243010183420991335640043434352506927103479838492662255348001536092257634115746278128486921186383228441

### Task 2: Public key
`RSA_2048_pub.der` is not an `RSAPublicKey`: it is a `SubjectPublicKeyInfo` – an algorithm identifier plus a `BIT STRING` that contains the `RSAPublicKey`
(see Task 2 in `02_openssl_keys` and Task 2 in `03_der_parsing`). Decode only the bytes inside the `BIT STRING`.

In [3]:
pub_der = open("RSA_2048_pub.der", "rb").read()
pub = rsa_spec.decode("RSAPublicKey", pub_der[24:])
print(pub)
assert pub["modulus"] == priv["modulus"] and pub["publicExponent"] == priv["publicExponent"]

{'modulus': 26600849061124703356336481747273244556424897312346608964949093451607513266591471026106943762750980853446024141708766672109725982003015999854187390472741283703452960170203172706692751708465051997402750571528164091286451527513148700346006515152876039065521975773899338302200230413838645117361941719273038177957810503270043548045850626165166299090599599549006397916460291285774324441974192993549251294218087652252222592259968754112161696463425633580783656942059887637967565538581666544274937726011810584986873918823482565805634098210715585036832816990966938486266344067154502293851198053748263931313802845262670239577559, 'publicExponent': 3}


### Task 3: What the fields mean
Each field of `RSAPrivateKey` is defined by a relation. Complete the checks, with $\lambda(n) = \mathrm{lcm}(p-1, q-1)$:
- `modulus` $n = p \cdot q$ (`prime1` = $p$, `prime2` = $q$)
- `privateExponent` $d$: $e \cdot d \equiv 1 \pmod{\lambda(n)}$
- `exponent1` $= d \bmod (p-1)$, `exponent2` $= d \bmod (q-1)$
- `coefficient` $= q^{-1} \bmod p$

Note: textbooks often use $\varphi(n) = (p-1)(q-1)$. Since $\lambda(n)$ divides $\varphi(n)$, a $d$ computed modulo $\varphi(n)$ also satisfies the check with $\lambda(n)$ –
so the $\lambda(n)$ check works for keys from any tool, while some tools compute $d$ modulo $\lambda(n)$ and then the $\varphi(n)$ check fails.

In [4]:
import math

n, e, d = priv["modulus"], priv["publicExponent"], priv["privateExponent"]
p, q = priv["prime1"], priv["prime2"]

assert n == p * q
assert (e * d) % math.lcm(p - 1, q - 1) == 1
assert priv["exponent1"] == d % (p - 1)
assert priv["exponent2"] == d % (q - 1)
assert priv["coefficient"] == pow(q, -1, p)
print("all checks passed")

all checks passed
